In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, concatenate_datasets, DatasetDict

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
from scipy.spatial.distance import pdist, squareform
import umap

In [ ]:
CHECKPOINT_BASE = "facebook/data2vec-audio-base-960h"
CHECKPOINT_FINETUNED  = "VasilisAsim/data2vec-finetuned-TESS_full"
SEED = 42

model_base = AutoModelForAudioClassification.from_pretrained(CHECKPOINT_BASE)
model_ft = AutoModelForAudioClassification.from_pretrained(CHECKPOINT_FINETUNED)
feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT_FINETUNED)

model_or.eval()
model_ft.eval()

###Φόρτωση dataset και speaker-independent split

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

### Εξαγωγή embeddings (mean pooling του τελευταίου επιπέδου) + κανονικοποίηση

In [ ]:
def extract_embeddings(model, dataset, feature_extractor):
    embeddings, emotions, speakers = [], [], []
    for sample in dataset:
        audio = sample["audio"]["array"]
        inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            hidden = model.data2vec_audio(inputs["input_values"]).last_hidden_state
        embeddings.append(hidden.mean(dim=1).squeeze().numpy())
        emotions.append(sample["emotion"])
        speakers.append(sample["file"].split("_")[0])      # "OAF" ή "YAF"
    return np.array(embeddings), np.array(emotions), np.array(speakers)


def prepare(emb):
    """Κεντράρισμα + L2 κανονικοποίηση (ώστε ευκλείδεια ≡ cosine απόσταση)."""
    emb = emb.astype(np.float64)
    emb = emb - emb.mean(axis=0, keepdims=True)
    return emb / np.linalg.norm(emb, axis=1, keepdims=True)


emb_base_raw, emotions_str, speakers_str = extract_embeddings(model_base, test_dataset, feature_extractor)
emb_ft_raw, _, _ = extract_embeddings(model_ft, test_dataset, feature_extractor)
emb_base = prepare(emb_base_raw)
emb_ft = prepare(emb_ft_raw)

le_emotion = LabelEncoder()
labels = le_emotion.fit_transform(emotions_str)
class_names = le_emotion.classes_
print(f"Δείγματα: {len(labels)} | Κλάσεις: {list(class_names)}")

### Βοηθητικές συναρτήσεις μετρικών

In [ ]:
# Όλες οι μετρικές εφαρμόζονται σε κεντραρισμένα + L2-κανονικοποιημένα embeddings (prepare),
# οπότε η cosine και η ευκλείδεια απόσταση δίνουν την ίδια διάταξη γειτόνων.
METRIC = "cosine"

def dunn_index(X, y, metric=METRIC):
    classes = np.unique(y)
    centroids = {c: X[y == c].mean(axis=0) for c in classes}

    # Μέγιστη ενδο-κλασική διάμετρος
    intra = [
        pdist(X[y == c], metric=metric).max()
        for c in classes if (y == c).sum() > 1
    ]
    max_intra = max(intra) if intra else 1e-9

    # Ελάχιστη δια-centroid απόσταση (ίδια metric με το intra)
    centroid_matrix = np.array(list(centroids.values()))
    inter_matrix = squareform(pdist(centroid_matrix, metric=metric))
    min_inter = inter_matrix[np.triu_indices_from(inter_matrix, k=1)].min()

    return min_inter / max_intra


def mean_intra_class_distance(X, y, metric=METRIC):
    """
    Μέση απόσταση ζευγών σημείων ΜΕΣΑ στην ίδια κλάση, ανά κλάση και συνολικά.
    """
    classes = np.unique(y)
    per_class = {}
    for c in classes:
        pts = X[y == c]
        per_class[c] = pdist(pts, metric=metric).mean() if len(pts) > 1 else np.nan
    overall = float(np.nanmean(list(per_class.values())))
    return per_class, overall


def inter_class_centroid_distance(X, y, metric=METRIC):
    """
    Πίνακας αποστάσεων μεταξύ centroids κλάσεων + μέση τιμή.
    """
    classes = np.unique(y)
    centroids = np.array([X[y == c].mean(axis=0) for c in classes])
    dist_matrix = squareform(pdist(centroids, metric=metric))
    mean_dist = float(dist_matrix[np.triu_indices_from(dist_matrix, k=1)].mean())
    return dist_matrix, mean_dist, classes


def knn_label_purity(X, y, k=10, metric=METRIC):
    """
    Για κάθε σημείο: ποσοστό των k πλησιέστερων γειτόνων (εκτός εαυτού) με το ίδιο label.
    Χρησιμοποιείται στον raw (υψηλών διαστάσεων) χώρο.
    """
    nn = NearestNeighbors(n_neighbors=k + 1, metric=metric).fit(X)
    _, idx = nn.kneighbors(X)
    idx = idx[:, 1:]
    same_label = (y[idx] == y[:, None])
    return float(same_label.mean())


def neighborhood_hit(X_proj, y, k=10, metric="euclidean"):
    """
    Ίδια λογική με knn_label_purity, αλλά στον 2D ΠΡΟΒΕΒΛΗΜΕΝΟ χώρο (PCA/t-SNE/UMAP).
    Δείχνει αν η προβολή διατηρεί τη συνοχή ετικετών τοπικά.
    """
    return knn_label_purity(X_proj, y, k=k, metric=metric)


def class_compactness(X, y, metric=METRIC):
    """
    Μέση απόσταση σημείων μιας κλάσης από το ΔΙΚΟ ΤΗΣ centroid, ανά κλάση και συνολικά.
    """
    classes = np.unique(y)
    per_class = {}
    for c in classes:
        pts = X[y == c]
        centroid = pts.mean(axis=0)
        if metric == "cosine":
            num = pts @ centroid
            denom = np.linalg.norm(pts, axis=1) * np.linalg.norm(centroid) + 1e-12
            dists = 1 - num / denom
        else:
            dists = np.linalg.norm(pts - centroid, axis=1)
        per_class[c] = float(dists.mean())
    overall = float(np.mean(list(per_class.values())))
    return per_class, overall


def clustering_purity(X, true_labels, n_clusters=None, random_state=SEED):
    """
    Purity ενός unsupervised k-means clustering σε σχέση με πραγματικές ετικέτες
    (π.χ. speaker_id ή corpus). Purity = (1/N) * Σ_clusters max_πλειοψηφικό_label.
    """
    if n_clusters is None:
        n_clusters = len(np.unique(true_labels))
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_assignments = km.fit_predict(X)

    N = len(true_labels)
    total = 0
    for cid in np.unique(cluster_assignments):
        members = true_labels[cluster_assignments == cid]
        if len(members) == 0:
            continue
        _, counts = np.unique(members, return_counts=True)
        total += counts.max()
    return total / N


def knn_embedding_accuracy(X, y, k=5, cv=5):
    """
    Cross-validated k-NN accuracy (μέσα στο YAF).
    """
    clf = KNeighborsClassifier(n_neighbors=k, metric=METRIC)
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    scores = cross_val_score(clf, X, y, cv=skf)
    return scores.mean(), scores.std()

###Αναφορά γεωμετρίας ανά *μοντέλο*

In [ ]:
def full_geometry_report(X, y, name):
    sil = silhouette_score(X, y, metric=METRIC)
    db = davies_bouldin_score(X, y)
    ch = calinski_harabasz_score(X, y)
    dunn = dunn_index(X, y)

    _, mean_intra = mean_intra_class_distance(X, y)
    _, mean_inter, _ = inter_class_centroid_distance(X, y)
    sep_ratio = mean_inter / mean_intra

    purity_knn = knn_label_purity(X, y, k=10)
    _, compactness = class_compactness(X, y)
    knn_m, knn_s = knn_embedding_accuracy(X, y)

    return {
        "Model": name,
        "Silhouette (↑)": round(sil, 4),
        "Davies-Bouldin (↓)": round(db, 4),
        "Calinski-Harabasz (↑)": round(ch, 2),
        "Dunn Index (↑)": round(dunn, 4),
        "Intra-class dist (↓)": round(mean_intra, 4),
        "Inter-class centroid dist (↑)": round(mean_inter, 4),
        "Separation ratio (↑)": round(sep_ratio, 4),
        "k-NN label purity, k=10 (↑)": round(purity_knn, 4),
        "Class compactness (↓)": round(compactness, 4),
        "kNN accuracy (5-fold)": f"{knn_m:.3f} ± {knn_s:.3f}",
    }

report = pd.DataFrame([
    full_geometry_report(emb_base, labels, "Base (ASR)"),
    full_geometry_report(emb_ft, labels, "Fine-tuned (TESS)"),
])
print(report.T.to_string(header=False))

### Speaker-cluster purity

In [ ]:
def load_tess_mixed_subset(n_per_speaker=200, seed=SEED):
    full_dataset = load_dataset("AbstractTTS/TESS", split="train")
    full_dataset = full_dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        return "OAF" if "OAF" in example["audio"]["path"] else "YAF"

    oaf = full_dataset.filter(lambda x: get_speaker(x) == "OAF").shuffle(seed=seed).select(range(n_per_speaker))
    yaf = full_dataset.filter(lambda x: get_speaker(x) == "YAF").shuffle(seed=seed).select(range(n_per_speaker))

    mixed = concatenate_datasets([oaf, yaf]).cast_column("audio", Audio(sampling_rate=16_000))
    return mixed

### Speaker probe στο μεικτό υποσύνολο

In [ ]:
mixed_dataset = load_tess_mixed_subset(n_per_speaker=200)

emb_base_mixed_raw, _, speakers_mixed_str = extract_embeddings(model_base, mixed_dataset, feature_extractor)
emb_ft_mixed_raw, _, _ = extract_embeddings(model_ft, mixed_dataset, feature_extractor)
speaker_labels_mixed = LabelEncoder().fit_transform(speakers_mixed_str)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

rows = []
for name, emb in [("Base (ASR)", emb_base_mixed_raw), ("Fine-tuned (TESS)", emb_ft_mixed_raw)]:
    X = prepare(emb)
    acc = cross_val_score(probe, X, speaker_labels_mixed, cv=skf)
    purity = clustering_purity(X, speaker_labels_mixed)
    rows.append({"Model": name,
                 "Speaker probe acc": f"{acc.mean():.3f} ± {acc.std():.3f}",
                 "Speaker k-means purity": round(purity, 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("Τυχαίο επίπεδο = 0.5")

###Πίνακας inter-class centroid distances (fine-tuned)

In [ ]:
dist_matrix, mean_inter_ft, classes_order = inter_class_centroid_distance(emb_ft, labels)
class_names_ordered = [class_names[c] for c in classes_order]

plt.figure(figsize=(8, 6))
sns.heatmap(dist_matrix, xticklabels=class_names_ordered, yticklabels=class_names_ordered,
            annot=True, fmt=".2f", cmap="viridis")
plt.title("Cosine απόσταση μεταξύ centroids (fine-tuned, κεντραρισμένα embeddings)")
plt.tight_layout()
plt.show()

### 7. Neighborhood Hit στις 2D προβολές

In [ ]:
pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(emb_ft)
tsne_2d = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(emb_ft)
umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                    metric="cosine", random_state=SEED).fit_transform(emb_ft)

nh_report = pd.DataFrame([
    {"Method": "Raw (768-d)", "Neighborhood Hit (k=10)": round(knn_label_purity(emb_ft, labels, k=10), 4)},
    {"Method": "PCA (2D)",   "Neighborhood Hit (k=10)": round(neighborhood_hit(pca_2d, labels, k=10), 4)},
    {"Method": "t-SNE (2D)", "Neighborhood Hit (k=10)": round(neighborhood_hit(tsne_2d, labels, k=10), 4)},
    {"Method": "UMAP (2D)",  "Neighborhood Hit (k=10)": round(neighborhood_hit(umap_2d, labels, k=10), 4)},
])
print(nh_report.to_string(index=False))

### Cross-speaker probe: train OAF → test YAF

In [ ]:
train_dataset = dataset["train"]
emb_ft_oaf_raw, emo_oaf, _ = extract_embeddings(model_ft, train_dataset, feature_extractor)
y_oaf = le_emotion.transform(emo_oaf)

def center_l2(emb):
    """Κεντράρισμα ανά ομιλητή + L2."""
    return prepare(emb)   # το prepare ήδη κεντράρει ανά σύνολο, δηλαδή ανά ομιλητή εδώ

clf = LogisticRegression(max_iter=2000)

# (α) κεντράρισμα μία φορά για όλο το σύνολο (χωρίς ανά-ομιλητή διόρθωση)
all_raw = np.vstack([emb_ft_oaf_raw, emb_ft_raw])
Xa = all_raw - all_raw.mean(0); Xa /= np.linalg.norm(Xa, axis=1, keepdims=True)
n = len(emb_ft_oaf_raw)
acc_global = clf.fit(Xa[:n], y_oaf).score(Xa[n:], labels)

# (β) κεντράρισμα ξεχωριστά ανά ομιλητή
acc_per_speaker = clf.fit(center_l2(emb_ft_oaf_raw), y_oaf).score(center_l2(emb_ft_raw), labels)

print(f"OAF→YAF, global centering:      {acc_global:.3f}")
print(f"OAF→YAF, per-speaker centering: {acc_per_speaker:.3f}")
print("Αναφορά: μέσα στο YAF kNN 5-fold = 0.957, head του μοντέλου = 0.741")